In [3]:
from groq import Groq
from dotenv import load_dotenv
import os
import json

load_dotenv()
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
print("Groq подключён!")

Groq подключён!


In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)
print(response.choices[0].message.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are an appointment scheduling assistant for a healthcare facility."},
        {"role": "user", "content": "I need to see a doctor next week."}
    ]
)
print(response.choices[0].message.content)

I'd be happy to help you schedule an appointment with one of our doctors. Can you please tell me what type of doctor you would like to see (e.g. primary care, specialist, etc.) and what your availability is like next week?

Also, do you have a preferred day and time for your appointment, or are you flexible? And do you have any specific concerns or issues that you would like to address during your appointment?


In [ ]:
import json

# Описание функции для модели
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_availability",
            "description": "Check available appointment slots for a given date",
            "parameters": {
                "type": "object",
                "properties": {
                    "date": {
                        "type": "string",
                        "description": "Date in YYYY-MM-DD format"
                    }
                },
                "required": ["date"]
            }
        }
    }
]

print("Функция описана!")

Функция описана!


In [ ]:
def check_availability(date):
    """Проверяет доступные слоты на дату"""
    
    # Пока это фейковые данные — потом подключим базу данных
    fake_schedule = {
        "2026-04-28": ["09:00", "11:00", "14:00"],
        "2026-04-29": ["10:00", "15:00"],
        "2026-04-30": []
    }
    
    slots = fake_schedule.get(date, [])
    
    if slots:
        return f"Available slots on {date}: {', '.join(slots)}"
    else:
        return f"No available slots on {date}"

# Тестируем
print(check_availability("2026-04-28"))
print(check_availability("2026-04-30"))
print(check_availability("2026-05-01"))

Available slots on 2026-04-28: 09:00, 11:00, 14:00
No available slots on 2026-04-30
No available slots on 2026-05-01


In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are an appointment scheduling assistant."},
        {"role": "user", "content": "Are there any slots on April 28, 2026?"}
    ],
    tools=tools
)

message = response.choices[0].message
print("Модель решила:", message.tool_calls)

Модель решила: [ChatCompletionMessageToolCall(id='bed06az9p', function=Function(arguments='{"date":"2026-04-28"}', name='check_availability'), type='function')]


In [ ]:
# 1. Достаём что модель попросила
tool_call = message.tool_calls[0]
function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

print(f"Модель хочет вызвать: {function_name}")
print(f"С аргументами: {arguments}")

# 2. Вызываем функцию
result = check_availability(arguments["date"])
print(f"Результат: {result}")

# 3. Отправляем результат обратно модели
final_response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are an appointment scheduling assistant."},
        {"role": "user", "content": "Are there any slots on April 28, 2026?"},
        message,
        {"role": "tool", "content": result, "tool_call_id": tool_call.id}
    ],
    tools=tools
)

print("\nОтвет пациенту:")
print(final_response.choices[0].message.content)

Модель хочет вызвать: check_availability
С аргументами: {'date': '2026-04-28'}
Результат: Available slots on 2026-04-28: 09:00, 11:00, 14:00

Ответ пациенту:
There are available slots on April 28, 2026, at 09:00, 11:00, and 14:00.


In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": "You are an appointment scheduling assistant."},
        {"role": "user", "content": "Do you have anything available on May 5, 2026?"}
    ],
    tools=tools
)

message = response.choices[0].message

if message.tool_calls:
    tool_call = message.tool_calls[0]
    arguments = json.loads(tool_call.function.arguments)
    result = check_availability(arguments["date"])
    
    final_response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are an appointment scheduling assistant."},
            {"role": "user", "content": "Do you have anything available on May 5, 2026?"},
            message,
            {"role": "tool", "content": result, "tool_call_id": tool_call.id}
        ],
        tools=tools
    )
    print("Ответ пациенту:")
    print(final_response.choices[0].message.content)
else:
    print("Модель ответила без функции:")
    print(message.content)

Ответ пациенту:
None


In [ ]:
print("Content:", final_response.choices[0].message.content)
print("Tool calls:", final_response.choices[0].message.tool_calls)

Content: None
Tool calls: [ChatCompletionMessageToolCall(id='kgc9mt8q3', function=Function(arguments='{"date":"2026-05-06"}', name='check_availability'), type='function')]


In [ ]:
system_prompt = """You are an appointment scheduling assistant.
When a function returns no available slots, tell the patient 
honestly and ask if they want to check a different date.
Do NOT automatically check other dates yourself."""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Do you have anything available on May 5, 2026?"}
    ],
    tools=tools
)

message = response.choices[0].message
tool_call = message.tool_calls[0]
arguments = json.loads(tool_call.function.arguments)
result = check_availability(arguments["date"])

final_response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Do you have anything available on May 5, 2026?"},
        message,
        {"role": "tool", "content": result, "tool_call_id": tool_call.id}
    ],
    tools=tools
)

print("Ответ пациенту:")
print(final_response.choices[0].message.content)

Ответ пациенту:
I apologize, but it seems that there are no available appointment slots on May 5, 2026. Would you like to check a different date?


In [ ]:
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "What should I take for a headache?"}
    ],
    tools=tools
)

message = response.choices[0].message

if message.tool_calls:
    print("Модель вызвала функцию (плохо!):")
    print(message.tool_calls)
else:
    print("Ответ пациенту:")
    print(message.content)


Ответ пациенту:
I'm not able to provide medical advice. If you're experiencing a headache, I recommend speaking with a healthcare professional for guidance on treatment options. Would you like to schedule an appointment with a doctor? If so, please let me know a date you would like to schedule the appointment, and I can check availability for you.
